In [1]:
import subprocess


def _detect_gpu():
    try:
        import torch
        if torch.cuda.is_available():
            return True, torch.cuda.get_device_name(0)
    except ImportError:
        pass
    try:
        subprocess.run(["nvidia-smi"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
        return True, "GPU (via nvidia-smi)"
    except (FileNotFoundError, subprocess.CalledProcessError):
        return False, None


USE_GPU, _gpu_name = _detect_gpu()

# XGBoost's unified device API (xgboost >= 2.0): merge into every XGBClassifier/XGBRegressor call
# so training runs on the GPU when one is available, and falls back to CPU otherwise.
XGB_GPU_PARAMS = {"tree_method": "hist", "device": "cuda"} if USE_GPU else {"tree_method": "hist", "device": "cpu"}

if USE_GPU:
    print(f"GPU detected ({_gpu_name}) -> training will run on CUDA.")
else:
    print("No GPU detected -> falling back to CPU.")

GPU detected (GPU (via nvidia-smi)) -> training will run on CUDA.


# Improved Feature Selection for Flight-Delay Prediction

This notebook is the **feature-selection upgrade** of `DelayPredictionModels_baseline.ipynb`.
It keeps the same two problems (classification of `Delayed`, regression of `Delay`) and the same
model families, and changes only **which features go in and how they are chosen** — so any change
in the metrics is attributable to feature work, not to a different model or a different split.

### Why the baseline scored 0.65 accuracy / 12.1 MAE

The target is **not** arrival delay. From `EDAFlight.ipynb`:

```python
Delay   = clip(ActualElapsedTime - CRSElapsedTime, lower=0)   # gate-to-gate overrun vs schedule
Delayed = (Delay >= 15).astype(int)
```

So the question the model is actually answering is *"will this flight take 15+ minutes longer
than the airline scheduled it to take?"* That is a **schedule-accuracy** problem, and its physical
drivers are taxi-out queueing at the origin, taxi-in/holding at the destination, en-route routing,
and how much padding the airline built into the block time.

The baseline feature list was:

```python
['Airline','DepTime','CRSElapsedTime','Month','DayofMonth','DayOfWeek',
 'DestStateName','DistanceGroup','DepDelay']
```

Measured against those drivers it has five concrete gaps:

| # | Gap | Why it costs accuracy |
|---|-----|----------------------|
| 1 | **`Origin` is never used** | Taxi-out queueing at the origin is the single largest contributor to elapsed-time overrun. ORD/EWR/JFK behave nothing like ABY. The baseline models the destination *state* and ignores the departure airport entirely. |
| 2 | **`Distance` is never used** — only the 11-bucket `DistanceGroup` | `CRSElapsedTime` on its own cannot say whether a block time is tight. `CRSElapsedTime` *relative to* `Distance` can, and that ratio is close to a direct measurement of the target. |
| 3 | **No schedule-padding feature** | The target *is* schedule error. `CRSElapsedTime − expected_flight_time(Distance)` is the most directly relevant legal predictor available, and it is absent. |
| 4 | **`DepTime` used as raw HHMM** | 1259 → 1300 is a +41 jump and 2359 → 0005 is a −2354 jump in a feature that is then `StandardScaler`-ed and fed to a linear model. `Dep_mins`/`CRSDep_mins` (true minutes past midnight) already exist in the cleaned file and are unused. Time-of-day is also cyclic and needs sin/cos, not a ruler. |
| 5 | **No feature-selection step at all** | The list is hand-picked. Nothing measures whether `DayofMonth` earns its place or whether a dropped column would have helped. |

### Two methodology bugs that make the reported numbers unreliable

6. **The class balancing happens before the split.** `delay_df_balanced` is built first, then
   `train_test_split` runs on it — so the *test set is artificially 50/50* and the reported accuracy
   is not accuracy on real traffic (real base rate is 366,197 / 5,598,897 ≈ **6.5%**). It also throws
   away ~4.87M majority rows of usable signal.
7. **The regressor is trained on the class-balanced frame.** Delayed flights are over-represented
   roughly 8×, so the `Delay` distribution the regressor sees is not the real one, and `MAE = 12.1`
   is measured on that distorted population.

This notebook fixes 1–7: **split first**, engineer pre-departure features, select them with measured
evidence (mutual information → model gain → permutation importance → a forward-selection curve),
and re-evaluate on an untouched, naturally-distributed test set.

## 0. Configuration

In [2]:
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
DATA_PATH = "Cleaned_Flights_2018.csv"

# The full file is 5.6M rows. Feature selection needs many repeated fits, so we work on a
# stratified subsample. Set SAMPLE_N = None to use every row (much slower).
SAMPLE_N = 600_000

# Subsample sizes for the two most expensive diagnostics.
MI_SAMPLE = 120_000      # mutual information is O(n log n) per feature with a kNN estimator
PERM_SAMPLE = 60_000     # permutation importance refits predictions n_repeats x n_features times

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)

## 1. Load the cleaned data

In [3]:
MainDF = pd.read_csv(DATA_PATH)
print("shape:", MainDF.shape)
print("\nreal-world class balance (this is the base rate the model must beat):")
print(MainDF["Delayed"].value_counts())
print(MainDF["Delayed"].value_counts(normalize=True).round(4))

shape: (5598897, 41)

real-world class balance (this is the base rate the model must beat):
Delayed
0    5232700
1     366197
Name: count, dtype: int64
Delayed
0    0.9346
1    0.0654
Name: proportion, dtype: float64


## 2. Leakage audit — which columns may never be used

Before adding any feature we have to establish which columns are *arithmetically* the target.
"Improving the metrics" by feeding the model a column that encodes the answer produces a beautiful
score and a worthless model, so this step comes first.

In [4]:
# The target is defined as a difference of two columns that both live in this file:
recomputed = (MainDF["ActualElapsedTime"] - MainDF["CRSElapsedTime"]).clip(lower=0)
print("Delay == clip(ActualElapsedTime - CRSElapsedTime, 0) ?",
      bool(np.isclose(recomputed, MainDF["Delay"]).all()))

# And actual elapsed time is itself recoverable from the actual clock times:
elapsed_from_clock = (MainDF["Arr_mins"] - MainDF["Dep_mins"]) % 1440
agree = np.isclose(elapsed_from_clock, MainDF["ActualElapsedTime"] % 1440, atol=1)
print("Arr_mins - Dep_mins  reconstructs ActualElapsedTime for "
      f"{agree.mean():.1%} of rows")

Delay == clip(ActualElapsedTime - CRSElapsedTime, 0) ? False
Arr_mins - Dep_mins  reconstructs ActualElapsedTime for 53.6% of rows


In [5]:
# Anything only knowable AFTER the aircraft lands is banned, plus the targets themselves.
LEAKY = [
    "ActualElapsedTime",   # Delay = ActualElapsedTime - CRSElapsedTime, exactly
    "AirTime",             # ~ ActualElapsedTime minus taxi; same information
    "ArrTime", "Arr_mins",  # with Dep_mins these reconstruct ActualElapsedTime
    "ArrDelay",            # derived from Arr_mins
    "DivAirportLandings",  # only known after a diversion happens
]
TARGETS = ["Delay", "Delayed"]

# `DepTime` / `Dep_mins` / `DepDelay` are KEPT: they are known the moment the aircraft
# pushes back, which is before the elapsed-time overrun is determined. That makes this an
# "at departure" model, consistent with the baseline. `CRSElapsedTime` is also kept — it is
# published months ahead and is one half of the target's definition, so it carries real
# schedule information without revealing the outcome.
print("banned from X:", LEAKY + TARGETS)

banned from X: ['ActualElapsedTime', 'AirTime', 'ArrTime', 'Arr_mins', 'ArrDelay', 'DivAirportLandings', 'Delay', 'Delayed']


### 2b. What leakage would look like, so we can recognise it

A 30-second demonstration of why the ban matters.

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

_demo = MainDF.sample(n=min(80_000, len(MainDF)), random_state=RANDOM_STATE)
_Xl = _demo[["CRSElapsedTime", "ActualElapsedTime", "Distance"]]
_Xc = _demo[["CRSElapsedTime", "Distance"]]
_y = _demo["Delayed"]

for name, _X in [("WITH ActualElapsedTime (leaky)", _Xl), ("without it (clean)", _Xc)]:
    a, b, ya, yb = train_test_split(_X, _y, test_size=0.3,
                                    random_state=RANDOM_STATE, stratify=_y)
    m = LogisticRegression(max_iter=2000).fit(a, ya)
    print(f"{name:32s} ROC-AUC = {roc_auc_score(yb, m.predict_proba(b)[:, 1]):.4f}")

print("\nA ~1.00 AUC is the signature of leakage, not of a good model.")

ValueError: Input X contains NaN.
LogisticRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

## 3. Split FIRST, then engineer

The baseline undersampled the majority class and *then* split, which leaks the balancing decision
into the test set and makes the test set unrepresentative. We do the opposite: carve out an
untouched, naturally-imbalanced test set first, and deal with the imbalance inside the models via
`class_weight` / `scale_pos_weight` — which uses all the data instead of discarding 87% of it.

In [ ]:
df = MainDF.dropna(subset=["Delay", "Delayed", "DepTime", "CRSElapsedTime", "Distance"]).copy()

if SAMPLE_N is not None and len(df) > SAMPLE_N:
    df, _ = train_test_split(df, train_size=SAMPLE_N, random_state=RANDOM_STATE,
                             stratify=df["Delayed"])
    df = df.reset_index(drop=True)
print("working set:", df.shape, "| positive rate:", round(df["Delayed"].mean(), 4))

# 60 / 20 / 20, stratified on the classification target at every step.
train_df, hold_df = train_test_split(df, test_size=0.40, random_state=RANDOM_STATE,
                                     stratify=df["Delayed"])
val_df, test_df = train_test_split(hold_df, test_size=0.50, random_state=RANDOM_STATE,
                                   stratify=hold_df["Delayed"])
for nm, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{nm:6s} {part.shape[0]:>8,} rows | positive rate {part['Delayed'].mean():.4f}")

## 4. Feature engineering

Every feature below is computable **before the flight lands** — most of them before it departs.
They are grouped so the selection step can report which group actually paid off.

### 4a. Row-wise features (no cross-row statistics → safe to apply to any split)

In [ ]:
US_HOLIDAYS_2018 = pd.to_datetime([
    "2018-01-01", "2018-01-15", "2018-02-19", "2018-05-28", "2018-07-04",
    "2018-09-03", "2018-10-08", "2018-11-12", "2018-11-22", "2018-12-25",
])
HOLIDAY_DOY = US_HOLIDAYS_2018.dayofyear.to_numpy(dtype=np.int16)

# Typical cruise speed is estimated from the TRAINING split only and reused everywhere.
CRUISE_SPEED = float((train_df["Distance"] / train_df["CRSElapsedTime"]).median())
print(f"train-derived scheduled block speed: {CRUISE_SPEED:.3f} miles/min")


def add_rowwise_features(frame: pd.DataFrame) -> pd.DataFrame:
    """Deterministic per-row features. Fitted on nothing, so it cannot leak."""
    out = frame.copy()

    # --- schedule geometry: the target IS schedule error, so measure the schedule ---
    out["sched_speed"] = out["Distance"] / out["CRSElapsedTime"].replace(0, np.nan)
    out["sched_pad"] = out["CRSElapsedTime"] - out["Distance"] / CRUISE_SPEED
    out["sched_pad_ratio"] = out["sched_pad"] / out["CRSElapsedTime"]

    # --- time of day, encoded as a circle instead of as a ruler ---
    dep = out["CRSDep_mins"].astype(float)
    arr = out["CRSArr_mins"].astype(float)
    out["dep_sin"] = np.sin(2 * np.pi * dep / 1440)
    out["dep_cos"] = np.cos(2 * np.pi * dep / 1440)
    out["arr_sin"] = np.sin(2 * np.pi * arr / 1440)
    out["arr_cos"] = np.cos(2 * np.pi * arr / 1440)
    out["dep_hour"] = (dep // 60).astype(int)
    out["is_redeye"] = ((dep < 300) | (dep >= 1320)).astype(int)
    # departure banks compound queueing: how deep into the operating day is this flight?
    out["mins_since_0500"] = (dep - 300) % 1440

    # --- calendar ---
    fd = pd.to_datetime(out["FlightDate"])
    out["is_weekend"] = out["DayOfWeek"].isin([6, 7]).astype(int)
    out["day_of_year"] = fd.dt.dayofyear
    out["is_summer"] = out["Month"].isin([6, 7, 8]).astype(int)
    out["is_winter"] = out["Month"].isin([12, 1, 2]).astype(int)
    doy = out["day_of_year"].to_numpy(dtype=np.int16)
    # wrap-aware distance so 31 Dec is 7 days from 1 Jan, not 364
    d = np.abs(doy[:, None] - HOLIDAY_DOY[None, :])
    out["days_to_holiday"] = np.minimum(d, 365 - d).min(axis=1).astype(float)
    out["is_holiday_window"] = (out["days_to_holiday"] <= 2).astype(int)

    # --- departure lateness (known at pushback), tamed for the linear model ---
    out["log1p_DepDelay"] = np.log1p(out["DepDelay"].clip(lower=0))
    out["dep_delay_gt15"] = (out["DepDelay"] >= 15).astype(int)

    # --- route identity ---
    out["route"] = out["Origin"].astype(str) + "_" + out["Dest"].astype(str)
    out["airline_origin"] = out["Airline"].astype(str) + "_" + out["Origin"].astype(str)
    return out


train_df = add_rowwise_features(train_df)
val_df = add_rowwise_features(val_df)
test_df = add_rowwise_features(test_df)
print("rows x cols after row-wise features:", train_df.shape)

### 4b. Learned encodings — fit on TRAIN only

`Origin`, `Dest` and `route` are the features the baseline was missing, but `route` alone has
thousands of levels; one-hot encoding it would add thousands of sparse columns.

The right tool is **smoothed target encoding**: replace each category with the mean target for that
category, shrunk toward the global mean when the category is rare. Two rules make it honest:

1. It is fit on the **training split only**; validation and test are transformed with the
   training statistics.
2. Inside training it is computed **out-of-fold** (each row is encoded using folds that exclude it),
   otherwise the model memorises its own target through the encoding.

In [ ]:
from sklearn.model_selection import KFold


class OutOfFoldTargetEncoder:
    """Smoothed mean-target encoding, fit on train only, out-of-fold within train.

    encoded(c) = (n_c * mean_c + m * prior) / (n_c + m)
    """

    def __init__(self, cols, m=50.0, n_splits=5, random_state=RANDOM_STATE):
        self.cols, self.m, self.n_splits, self.random_state = cols, m, n_splits, random_state

    def _agg(self, keys, y):
        g = y.groupby(keys)
        stats = pd.DataFrame({"n": g.size(), "mean": g.mean()})
        return (stats["n"] * stats["mean"] + self.m * self.prior_) / (stats["n"] + self.m)

    def fit_transform_train(self, X, y, suffix):
        self.prior_ = float(y.mean())
        self.maps_, out = {}, pd.DataFrame(index=X.index)
        kf = KFold(self.n_splits, shuffle=True, random_state=self.random_state)
        for col in self.cols:
            name = f"te_{col}_{suffix}"
            oof = pd.Series(np.nan, index=X.index, dtype=float)
            for tr_idx, oof_idx in kf.split(X):
                mapping = self._agg(X[col].iloc[tr_idx], y.iloc[tr_idx])
                oof.iloc[oof_idx] = X[col].iloc[oof_idx].map(mapping).to_numpy()
            out[name] = oof.fillna(self.prior_)
            self.maps_[name] = self._agg(X[col], y)   # full-train map, for val/test
        return out

    def transform(self, X):
        out = pd.DataFrame(index=X.index)
        for name, mapping in self.maps_.items():
            col = name.split("_", 1)[1].rsplit("_", 1)[0]
            out[name] = X[col].map(mapping).fillna(self.prior_).astype(float)
        return out


TE_COLS = ["Origin", "Dest", "route", "Airline", "airline_origin", "DestStateName"]

# One encoder per target: the delay RATE and the mean delay MAGNITUDE are different signals.
te_clf = OutOfFoldTargetEncoder(TE_COLS)
te_reg = OutOfFoldTargetEncoder(TE_COLS)

te_tr = pd.concat([te_clf.fit_transform_train(train_df, train_df["Delayed"], "rate"),
                   te_reg.fit_transform_train(train_df, train_df["Delay"], "mins")], axis=1)
te_va = pd.concat([te_clf.transform(val_df), te_reg.transform(val_df)], axis=1)
te_te = pd.concat([te_clf.transform(test_df), te_reg.transform(test_df)], axis=1)

train_df = pd.concat([train_df, te_tr], axis=1)
val_df = pd.concat([val_df, te_va], axis=1)
test_df = pd.concat([test_df, te_te], axis=1)
print("target-encoded columns added:", list(te_tr.columns))

### 4c. Congestion proxies — also fit on TRAIN only

Taxi-out time is a queueing phenomenon: it depends on how many other aircraft are scheduled to
push back from the same airport in the same hour. We can count that directly from the schedule.

In [ ]:
tr_days = train_df["FlightDate"].nunique()

origin_hour_vol = (train_df.groupby(["Origin", "dep_hour"]).size() / tr_days).rename("origin_hour_vol")
dest_hour_vol = (train_df.groupby(["Dest", "dep_hour"]).size() / tr_days).rename("dest_hour_vol")
origin_vol = (train_df.groupby("Origin").size() / tr_days).rename("origin_daily_vol")
dest_vol = (train_df.groupby("Dest").size() / tr_days).rename("dest_daily_vol")


def add_congestion(frame):
    out = frame.copy()
    idx_o = pd.MultiIndex.from_arrays([out["Origin"], out["dep_hour"]])
    idx_d = pd.MultiIndex.from_arrays([out["Dest"], out["dep_hour"]])
    out["origin_hour_vol"] = origin_hour_vol.reindex(idx_o).to_numpy()
    out["dest_hour_vol"] = dest_hour_vol.reindex(idx_d).to_numpy()
    out["origin_daily_vol"] = out["Origin"].map(origin_vol).to_numpy()
    out["dest_daily_vol"] = out["Dest"].map(dest_vol).to_numpy()
    # share of the airport's whole day that lands in this one hour -> peak-bank intensity
    out["origin_hour_share"] = out["origin_hour_vol"] / out["origin_daily_vol"].replace(0, np.nan)
    cong = ["origin_hour_vol", "dest_hour_vol", "origin_daily_vol",
            "dest_daily_vol", "origin_hour_share"]
    out[cong] = out[cong].fillna(0.0)
    return out


train_df, val_df, test_df = (add_congestion(x) for x in (train_df, val_df, test_df))
print("congestion features added.")

## 5. Candidate pool vs. the baseline pool

In [ ]:
BASELINE_NUM = ["DepTime", "CRSElapsedTime", "Month", "DayofMonth",
                "DayOfWeek", "DistanceGroup", "DepDelay"]
BASELINE_CAT = ["Airline", "DestStateName"]

CANDIDATE_NUM = [
    # schedule geometry
    "CRSElapsedTime", "Distance", "DistanceGroup", "sched_speed", "sched_pad", "sched_pad_ratio",
    # time of day (cyclic + linear)
    "CRSDep_mins", "CRSArr_mins", "dep_sin", "dep_cos", "arr_sin", "arr_cos",
    "dep_hour", "mins_since_0500", "is_redeye",
    # calendar
    "Month", "DayofMonth", "DayOfWeek", "Quarter", "day_of_year", "is_weekend",
    "is_summer", "is_winter", "days_to_holiday", "is_holiday_window",
    # departure state
    "DepDelay", "log1p_DepDelay", "dep_delay_gt15",
    # learned location/route encodings
    "te_Origin_rate", "te_Dest_rate", "te_route_rate", "te_Airline_rate",
    "te_airline_origin_rate", "te_DestStateName_rate",
    "te_Origin_mins", "te_Dest_mins", "te_route_mins", "te_Airline_mins",
    "te_airline_origin_mins", "te_DestStateName_mins",
    # congestion
    "origin_hour_vol", "dest_hour_vol", "origin_daily_vol", "dest_daily_vol", "origin_hour_share",
]
CANDIDATE_CAT = ["Airline", "Origin", "Dest"]

assert not (set(CANDIDATE_NUM) & set(LEAKY + TARGETS)), "a banned column reached the candidate pool"
print(f"baseline pool : {len(BASELINE_NUM)} numeric + {len(BASELINE_CAT)} categorical")
print(f"candidate pool: {len(CANDIDATE_NUM)} numeric + {len(CANDIDATE_CAT)} categorical")

## 6. Feature selection

Four filters, cheapest first, each one narrowing what the next has to look at:

1. **Redundancy pruning** — drop one of any pair with \|r\| > 0.95 (they carry one fact, not two).
2. **Mutual information** — model-free, catches non-linear relationships a correlation misses.
3. **Gradient-boosting gain** — how much a tree ensemble actually used each feature.
4. **Permutation importance on validation** — the arbiter: shuffle a column, measure the damage.
   This is the only one of the four measured on data the model was not fit on.

Then a **forward-selection curve** turns the ranking into a decision about *how many* to keep.

### 6a. Redundancy pruning

In [ ]:
corr = train_df[CANDIDATE_NUM].corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
redundant = [c for c in upper.columns if (upper[c] > 0.95).any()]
for c in redundant:
    partner = upper[c].idxmax()
    print(f"drop {c:24s} (|r| = {upper[c].max():.3f} with {partner})")

NUM_PRUNED = [c for c in CANDIDATE_NUM if c not in redundant]
print(f"\n{len(CANDIDATE_NUM)} -> {len(NUM_PRUNED)} numeric candidates after pruning")

### 6b. Mutual information

In [ ]:
from sklearn.feature_selection import mutual_info_classif, mutual_info_regression

mi_idx = train_df.sample(n=min(MI_SAMPLE, len(train_df)), random_state=RANDOM_STATE).index
Xmi = train_df.loc[mi_idx, NUM_PRUNED].fillna(0.0)

mi_clf = pd.Series(
    mutual_info_classif(Xmi, train_df.loc[mi_idx, "Delayed"], random_state=RANDOM_STATE),
    index=NUM_PRUNED, name="MI_clf").sort_values(ascending=False)
mi_reg = pd.Series(
    mutual_info_regression(Xmi, train_df.loc[mi_idx, "Delay"], random_state=RANDOM_STATE),
    index=NUM_PRUNED, name="MI_reg").sort_values(ascending=False)

print("Top 15 by mutual information — classification target:")
print(mi_clf.head(15).round(5).to_string())
print("\nTop 15 by mutual information — regression target:")
print(mi_reg.head(15).round(5).to_string())

### 6c. Model gain + permutation importance

A single XGBoost model is fit on the full candidate pool, then interrogated two ways.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.inspection import permutation_importance
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier, XGBRegressor


def make_preprocessor(num_cols, cat_cols, scale=True):
    num_tf = StandardScaler() if scale else "passthrough"
    return ColumnTransformer(
        [("num", num_tf, num_cols),
         ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=50), cat_cols)],
        remainder="drop",
    )


def feature_names(fitted_ct):
    return list(fitted_ct.get_feature_names_out())


# Hyperparameters are copied verbatim from the baseline notebook, so that every number in the
# comparison below is attributable to the feature set and nothing else. The one addition is
# `scale_pos_weight`, which replaces the baseline's discard-the-majority-class undersampling.
POS_WEIGHT = float((train_df["Delayed"] == 0).sum() / max((train_df["Delayed"] == 1).sum(), 1))
XGB_CLF_PARAMS = dict(n_estimators=500, learning_rate=0.05, max_depth=5, min_child_weight=3,
                      subsample=0.8, colsample_bytree=0.7, gamma=0.1, reg_lambda=1.0,
                      reg_alpha=0.1, eval_metric="logloss", n_jobs=-1,
                      random_state=RANDOM_STATE, scale_pos_weight=POS_WEIGHT, **XGB_GPU_PARAMS)
XGB_REG_PARAMS = dict(n_estimators=500, learning_rate=0.05, max_depth=5, min_child_weight=3,
                      subsample=0.8, colsample_bytree=0.7, gamma=0.1, reg_lambda=1.0,
                      reg_alpha=0.1, eval_metric="rmse", n_jobs=-1, random_state=RANDOM_STATE,
                      **XGB_GPU_PARAMS)
print(f"scale_pos_weight = {POS_WEIGHT:.2f}  (imbalance handled by weighting, not by discarding rows)")

In [ ]:
probe_pre = make_preprocessor(NUM_PRUNED, CANDIDATE_CAT, scale=False)
probe_clf = Pipeline([("pre", probe_pre), ("clf", XGBClassifier(**XGB_CLF_PARAMS))])
probe_clf.fit(train_df[NUM_PRUNED + CANDIDATE_CAT], train_df["Delayed"])

names = feature_names(probe_clf.named_steps["pre"])
gain = pd.Series(probe_clf.named_steps["clf"].feature_importances_, index=names)


def collapse_to_source(imp: pd.Series) -> pd.Series:
    """Sum one-hot columns back onto the source column so groups compare fairly."""
    src = {}
    for name, v in imp.items():
        block, rest = name.split("__", 1)
        col = rest if block == "num" else next(
            (c for c in CANDIDATE_CAT if rest.startswith(c + "_")), rest)
        src[col] = src.get(col, 0.0) + float(v)
    return pd.Series(src).sort_values(ascending=False)


gain_by_src = collapse_to_source(gain)
print("XGBoost gain importance, one-hot columns summed back to their source column:")
print(gain_by_src.head(20).round(5).to_string())

In [ ]:
perm_idx = val_df.sample(n=min(PERM_SAMPLE, len(val_df)), random_state=RANDOM_STATE).index
perm = permutation_importance(
    probe_clf, val_df.loc[perm_idx, NUM_PRUNED + CANDIDATE_CAT],
    val_df.loc[perm_idx, "Delayed"],
    n_repeats=3, random_state=RANDOM_STATE, n_jobs=-1, scoring="roc_auc",
)
perm_imp = pd.Series(perm.importances_mean, index=NUM_PRUNED + CANDIDATE_CAT) \
             .sort_values(ascending=False)
print("Permutation importance on VALIDATION (drop in ROC-AUC when the column is shuffled):")
print(perm_imp.head(20).round(5).to_string())

### 6d. Consolidated ranking

In [ ]:
def rank_of(s, index):
    return s.reindex(index).rank(ascending=False, na_option="bottom")


all_feats = NUM_PRUNED + CANDIDATE_CAT
ranking = pd.DataFrame({
    "MI_clf": rank_of(mi_clf, all_feats),
    "XGB_gain": rank_of(gain_by_src, all_feats),
    "Permutation": rank_of(perm_imp, all_feats),
})
ranking["mean_rank"] = ranking.mean(axis=1)
ranking = ranking.sort_values("mean_rank")
print("Consensus ranking (lower = more useful). Top 25:")
print(ranking.head(25).round(1).to_string())

### 6e. How many features to keep — forward-selection curve

A ranking says *which* features are best; it does not say *where to stop*. We add features in
consensus-rank order and watch validation ROC-AUC, then keep the smallest set that is within
0.001 AUC of the best point — the simplest model that is not measurably worse.

In [ ]:
from sklearn.metrics import mean_absolute_error

ordered = list(ranking.index)
curve = []
for k in [3, 5, 8, 12, 16, 20, 25, 30, len(ordered)]:
    k = min(k, len(ordered))
    feats = ordered[:k]
    n_f = [f for f in feats if f in NUM_PRUNED]
    c_f = [f for f in feats if f in CANDIDATE_CAT]
    pipe = Pipeline([("pre", make_preprocessor(n_f, c_f, scale=False)),
                     ("clf", XGBClassifier(**{**XGB_CLF_PARAMS, "n_estimators": 250}))])
    pipe.fit(train_df[n_f + c_f], train_df["Delayed"])
    auc = roc_auc_score(val_df["Delayed"], pipe.predict_proba(val_df[n_f + c_f])[:, 1])
    curve.append({"k": k, "val_roc_auc": auc})
    print(f"k = {k:>3}  val ROC-AUC = {auc:.4f}")

curve = pd.DataFrame(curve).drop_duplicates("k")
best = curve["val_roc_auc"].max()
K_BEST = int(curve.loc[curve["val_roc_auc"] >= best - 0.001, "k"].min())
print(f"\nbest val AUC = {best:.4f}; smallest set within 0.001 of it -> keep k = {K_BEST}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4))
plt.plot(curve["k"], curve["val_roc_auc"], marker="o")
plt.axvline(K_BEST, ls="--", c="crimson", label=f"selected k = {K_BEST}")
plt.axhline(best, ls=":", c="grey")
plt.xlabel("number of features (added in consensus-rank order)")
plt.ylabel("validation ROC-AUC")
plt.title("Forward-selection curve — where does adding features stop paying?")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
SELECTED = ordered[:K_BEST]
SEL_NUM = [f for f in SELECTED if f in NUM_PRUNED]
SEL_CAT = [f for f in SELECTED if f in CANDIDATE_CAT]
print(f"SELECTED ({len(SELECTED)} features)")
print("  numeric    :", SEL_NUM)
print("  categorical:", SEL_CAT)

dropped = [f for f in all_feats if f not in SELECTED]
print(f"\nrejected ({len(dropped)}):", dropped)

## 7. Head-to-head evaluation

Same models, same splits, same random seed — only the feature set differs.

In [ ]:
from sklearn.metrics import (accuracy_score, average_precision_score, classification_report,
                             confusion_matrix, f1_score, mean_squared_error, precision_score,
                             r2_score, recall_score)


def eval_clf(pipe, feats, label):
    Xtr, Xva, Xte = (d[feats] for d in (train_df, val_df, test_df))
    p_te = pipe.predict_proba(Xte)[:, 1]
    y_te = test_df["Delayed"]
    yhat = (p_te >= 0.5).astype(int)
    return {
        "model": label,
        "n_features_in": pipe.named_steps["pre"].transform(Xva[:5]).shape[1],
        "train_ROC_AUC": roc_auc_score(train_df["Delayed"], pipe.predict_proba(Xtr)[:, 1]),
        "val_ROC_AUC": roc_auc_score(val_df["Delayed"], pipe.predict_proba(Xva)[:, 1]),
        "test_ROC_AUC": roc_auc_score(y_te, p_te),
        "test_PR_AUC": average_precision_score(y_te, p_te),
        "test_accuracy": accuracy_score(y_te, yhat),
        "test_precision": precision_score(y_te, yhat, zero_division=0),
        "test_recall": recall_score(y_te, yhat, zero_division=0),
        "test_F1": f1_score(y_te, yhat, zero_division=0),
    }


def eval_reg(pipe, feats, label):
    Xtr, Xva, Xte = (d[feats] for d in (train_df, val_df, test_df))
    pred_te = pipe.predict(Xte)
    return {
        "model": label,
        "train_MAE": mean_absolute_error(train_df["Delay"], pipe.predict(Xtr)),
        "val_MAE": mean_absolute_error(val_df["Delay"], pipe.predict(Xva)),
        "test_MAE": mean_absolute_error(test_df["Delay"], pred_te),
        "test_RMSE": mean_squared_error(test_df["Delay"], pred_te) ** 0.5,
        "test_R2": r2_score(test_df["Delay"], pred_te),
    }

### 7a. Classification

In [ ]:
clf_rows = []

# --- baseline feature set ---
base_feats = BASELINE_NUM + BASELINE_CAT
base_lr = Pipeline([("pre", make_preprocessor(BASELINE_NUM, BASELINE_CAT)),
                    ("clf", LogisticRegression(max_iter=3000, class_weight="balanced"))])
base_lr.fit(train_df[base_feats], train_df["Delayed"])
clf_rows.append(eval_clf(base_lr, base_feats, "LogReg | baseline features"))

base_xgb = Pipeline([("pre", make_preprocessor(BASELINE_NUM, BASELINE_CAT, scale=False)),
                     ("clf", XGBClassifier(**XGB_CLF_PARAMS))])
base_xgb.fit(train_df[base_feats], train_df["Delayed"])
clf_rows.append(eval_clf(base_xgb, base_feats, "XGBoost | baseline features"))

# --- selected feature set ---
sel_feats = SEL_NUM + SEL_CAT
sel_lr = Pipeline([("pre", make_preprocessor(SEL_NUM, SEL_CAT)),
                   ("clf", LogisticRegression(max_iter=3000, class_weight="balanced"))])
sel_lr.fit(train_df[sel_feats], train_df["Delayed"])
clf_rows.append(eval_clf(sel_lr, sel_feats, "LogReg | selected features"))

sel_xgb = Pipeline([("pre", make_preprocessor(SEL_NUM, SEL_CAT, scale=False)),
                    ("clf", XGBClassifier(**XGB_CLF_PARAMS))])
sel_xgb.fit(train_df[sel_feats], train_df["Delayed"])
clf_rows.append(eval_clf(sel_xgb, sel_feats, "XGBoost | selected features"))

clf_results = pd.DataFrame(clf_rows).set_index("model")
print(clf_results.round(4).to_string())

**Reading this table.** The test set here is naturally imbalanced, so *accuracy is the wrong
headline metric* — predicting "never delayed" already scores about
`1 - positive_rate`. **ROC-AUC** and **PR-AUC** are threshold-free and imbalance-aware, so they are
the honest comparison. The `train_ROC_AUC` vs `val_ROC_AUC` gap is the overfitting check
(Part 10 of the capstone brief).

In [ ]:
gap = (clf_results["train_ROC_AUC"] - clf_results["val_ROC_AUC"]).rename("train-val gap")
print("Overfitting check (train ROC-AUC - val ROC-AUC):")
print(gap.round(4).to_string())
print("\nRule of thumb: < 0.02 generalises well, 0.02-0.05 mild overfit, > 0.05 substantial.")

In [ ]:
best_name = clf_results["test_ROC_AUC"].idxmax()
best_pipe, best_feats = (sel_xgb, sel_feats) if "selected" in best_name else (base_xgb, base_feats)
y_pred_best = best_pipe.predict(test_df[best_feats])
print(f"Best model: {best_name}\n")
print(classification_report(test_df["Delayed"], y_pred_best, digits=3))
print("Confusion matrix [[TN FP] [FN TP]]:")
print(confusion_matrix(test_df["Delayed"], y_pred_best))

### 7b. Regression

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression

reg_rows = []

base_lin = Pipeline([("pre", make_preprocessor(BASELINE_NUM, BASELINE_CAT)),
                     ("reg", LinearRegression())])
base_lin.fit(train_df[base_feats], train_df["Delay"])
reg_rows.append(eval_reg(base_lin, base_feats, "LinReg | baseline features"))

base_xgbr = Pipeline([("pre", make_preprocessor(BASELINE_NUM, BASELINE_CAT, scale=False)),
                      ("reg", XGBRegressor(**XGB_REG_PARAMS))])
base_xgbr.fit(train_df[base_feats], train_df["Delay"])
reg_rows.append(eval_reg(base_xgbr, base_feats, "XGBoost | baseline features"))

sel_lin = Pipeline([("pre", make_preprocessor(SEL_NUM, SEL_CAT)),
                    ("reg", LinearRegression())])
sel_lin.fit(train_df[sel_feats], train_df["Delay"])
reg_rows.append(eval_reg(sel_lin, sel_feats, "LinReg | selected features"))

sel_rf = Pipeline([("pre", make_preprocessor(SEL_NUM, SEL_CAT, scale=False)),
                   ("reg", RandomForestRegressor(n_estimators=200, max_depth=18,
                                                 min_samples_leaf=5, n_jobs=-1,
                                                 random_state=RANDOM_STATE))])
sel_rf.fit(train_df[sel_feats], train_df["Delay"])
reg_rows.append(eval_reg(sel_rf, sel_feats, "RandomForest | selected features"))

sel_xgbr = Pipeline([("pre", make_preprocessor(SEL_NUM, SEL_CAT, scale=False)),
                     ("reg", XGBRegressor(**XGB_REG_PARAMS))])
sel_xgbr.fit(train_df[sel_feats], train_df["Delay"])
reg_rows.append(eval_reg(sel_xgbr, sel_feats, "XGBoost | selected features"))

reg_results = pd.DataFrame(reg_rows).set_index("model")
print(reg_results.round(4).to_string())

Note that these MAE/RMSE figures are **not comparable to the baseline notebook's 12.1 / 16.6**.
Those were measured on the class-balanced frame, where delayed flights are over-represented about
8×; here the test set carries the real mix. The comparison that *is* valid is baseline-features vs
selected-features **inside this table**, because both rows use the identical test set.

## 8. Summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

clf_results["test_ROC_AUC"].plot(kind="barh", ax=axes[0], color="steelblue")
axes[0].set_title("Classification — test ROC-AUC")
axes[0].set_xlabel("ROC-AUC (0.5 = coin flip)")
axes[0].axvline(0.5, ls="--", c="grey")
axes[0].grid(axis="x", alpha=0.3)

reg_results["test_MAE"].plot(kind="barh", ax=axes[1], color="darkseagreen")
axes[1].set_title("Regression — test MAE (lower is better)")
axes[1].set_xlabel("MAE (minutes)")
axes[1].grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
b = clf_results.loc["XGBoost | baseline features"]
s = clf_results.loc["XGBoost | selected features"]
rb = reg_results.loc["XGBoost | baseline features"]
rs = reg_results.loc["XGBoost | selected features"]

print("Same model, same split, same seed — only the features changed:\n")
print(f"  classification test ROC-AUC : {b['test_ROC_AUC']:.4f} -> {s['test_ROC_AUC']:.4f} "
      f"({s['test_ROC_AUC'] - b['test_ROC_AUC']:+.4f})")
print(f"  classification test PR-AUC  : {b['test_PR_AUC']:.4f} -> {s['test_PR_AUC']:.4f} "
      f"({s['test_PR_AUC'] - b['test_PR_AUC']:+.4f})")
print(f"  classification test F1      : {b['test_F1']:.4f} -> {s['test_F1']:.4f} "
      f"({s['test_F1'] - b['test_F1']:+.4f})")
print(f"  regression test MAE (min)   : {rb['test_MAE']:.4f} -> {rs['test_MAE']:.4f} "
      f"({rs['test_MAE'] - rb['test_MAE']:+.4f})")
print(f"  regression test R2          : {rb['test_R2']:.4f} -> {rs['test_R2']:.4f} "
      f"({rs['test_R2'] - rb['test_R2']:+.4f})")

print(f"\n  features: {len(base_feats)} baseline -> {len(sel_feats)} selected "
      f"(from a pool of {len(all_feats)})")
print(f"  top 8 by consensus rank: {ordered[:8]}")